In [ ]:
import scqubits
import numpy as np
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from tqdm import tqdm 
import pickle
from matplotlib.colors import  LogNorm
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import matplotlib.gridspec as gridspec


import sys
sys.path.append('../')
from utils_models import *


def plot_fluxonium_transitions(ax,matrix, energies,xlim,ylim = (2e-3,0.8)):
    from random import random 
    k = len(energies)  # Assuming matrix is k x k and energies is a list of length k

    label_positions = []

    for i in range(3):
        for j in range(k): 
            if j >i:
                freq_ij = np.abs(energies[i] - energies[j])  # Frequency of the transition
                element_ij = abs(matrix[i, j])  # Matrix element

                ax.plot([freq_ij, freq_ij], [0, element_ij], marker='o', color='grey', markersize=4)

                if element_ij >= ylim[0]:
                    x_loc = freq_ij  - 0.1
                    y_loc = element_ij+0.003
                    text_pos = (x_loc, y_loc)


                    if x_loc < xlim[-1] and x_loc > xlim[0] and y_loc > ylim[0] and y_loc <ylim[-1]:
                        ax.text(*(x_loc,y_loc), rf'{i}-{j}', size=9)
                        label_positions.append(text_pos)
                    else:
                        print(f'omitted: {i}-{j}')
    plt.text(-0.07, 1, '(a)', transform=plt.gca().transAxes, fontsize=12, fontweight='bold', va='top', color='black')

    ax.set_ylabel(r'$\langle i | \hat{n} | j \rangle$')
    # ax.set_xlabel(r'transition frequency $\omega_{ij}$')
    ax.grid(which='major', linestyle=':', linewidth='0.5', color='black')
    ax.set_xlim(xlim)
    # ax.set_yscale('log')
    # ax.set_ylim(ylim)



def find_closest_using_numpy(l, f):
    arr = np.array(l)
    idx = np.searchsorted(arr, f, side='left')
    idx = np.clip(idx, 1, len(arr) - 1)
    return idx - 1 if abs(arr[idx - 1] - f) <= abs(arr[idx] - f) else idx



colors = [(0.7, 0.6, 0.0), (0.5*0.5, 0.7*0.5, 0.5*0.5), (0.7, 0.6, 0.7)]
linestyles = [(0,(3,1,1,1)),'-',(0,(5,2,5,2))]


def plot_sweep_Er(ax,elements, evals,Er_list,qls = [0,1,2,3],symlog = True,g=0.1):
    num_evals =len(evals)

    y_max = 10
    y_min = -10
    for ql , color, linestyle in zip(qls, colors, linestyles):
        shift_from_qubit_transition = []
        for Er in tqdm(Er_list, desc = "Er loop"):
            shifts = [get_shift_accurate(elements[ql,ql2], evals[ql2], evals[ql], Er) for ql2 in range(num_evals)] 
            if not symlog:
                shift_from_qubit_transition.append(abs(sum(shifts)))
            else:
                shift_from_qubit_transition.append(sum(shifts))
        ax.plot(Er_list, shift_from_qubit_transition, label=rf'$\chi_{ql}$',color = color, linestyle = linestyle)

        # # Plot the label of transition on the tip of poles
        # for ql2 in range(ql+1,12):
        #     transition =  abs(evals[ql2]-evals[ql])
        #     if abs(elements[ql,ql2]) > 1e-5 and transition < Er_list[-1] and transition > Er_list[0]:
        #         Er_index = find_closest_using_numpy(Er_list, transition)
        #         y_pos = shift_from_qubit_transition[Er_index]
        #         y_pos = min(y_pos, y_max)
        #         y_pos = max(y_pos, y_min)
        #         plt.text(transition, y_pos, f'{ql}-{ql2}', fontsize=8, ha='center', va='bottom')


    plt.text(-0.07, 1, '(b)', transform=plt.gca().transAxes, fontsize=12, fontweight='bold', va='top', color='black')
    ax.grid(which='major', linestyle=':', linewidth='0.5', color='black')
    ax.minorticks_on()
    ax.grid(which='minor', linestyle=':', linewidth='0.5', color='gray')
    ax.set_xlim(Er_list[0],Er_list[-1])
    if not symlog:
        ax.set_yscale('log')
        ax.set_ylim(1e-2,y_max)
    else:
        ax.set_yscale('symlog', linthresh=1e-2,linscale=0.1)
        ax.set_ylim(y_min,y_max)
    
    ax.set_xlabel(rf'$\omega_r$')
    ax.set_ylabel(r'$\chi_{j}$')
    ax.legend(loc = 'upper left')


from typing import List
def dressed_transition_frequency_over_2pi(hilbertspace,s0: List[int], s1: List[int]) -> float:
    s0 = hilbertspace.dressed_index(s0)
    s1 = hilbertspace.dressed_index(s1)
    return abs(hilbertspace.energy_by_dressed_index(s1) - hilbertspace.energy_by_dressed_index(s0))

def replace_non_float64_with_none(lst):
    for i in range(len(lst)):
        if type(lst[i]) is not np.float64:
            lst[i] = None
    return lst



def get_chi_lists(qbt,Er_list, g= 0.13):
    chi0_list = []
    chi1_list = []
    chi2_list = []
    chi3_list = []
    for Er in tqdm(Er_list,desc='iterating over Er'):
        osc = scqubits.Oscillator(
            E_osc=Er,
            truncated_dim=5
        )
        hilbertspace = scqubits.HilbertSpace([qbt, osc])
        hilbertspace.add_interaction(
            g_strength=g,
            op1=qbt.n_operator,
            op2=osc.creation_operator,
            add_hc=True
        )
        hilbertspace.generate_lookup()
        chi0 = dressed_transition_frequency_over_2pi(hilbertspace,(0,0),(0,1)) - Er
        chi1 = dressed_transition_frequency_over_2pi(hilbertspace,(1,0),(1,1)) - Er
        chi2 = dressed_transition_frequency_over_2pi(hilbertspace,(2,0),(2,1)) - Er

        chi0_list.append(chi0)
        chi1_list.append(chi1)
        chi2_list.append(chi2)

    chi0_list = replace_non_float64_with_none(chi0_list)
    chi1_list = replace_non_float64_with_none(chi1_list)
    chi2_list = replace_non_float64_with_none(chi2_list)
    return chi0_list,chi1_list,chi2_list

def plot_sweep_Er_diagonalization(ax,chi0_list,chi1_list,chi2_list,Er_list):
    for ql, chi_list , color, linestyle in zip([0,1,2],[chi0_list,chi1_list,chi2_list], colors, linestyles):
        ax.plot(Er_list, chi_list, label=rf'$\chi_{ql}$',color = color, linestyle = linestyle)

    plt.text(-0.07, 1, '(b)', transform=plt.gca().transAxes, fontsize=12, fontweight='bold', va='top', color='black')
    ax.grid(which='major', linestyle=':', linewidth='0.5', color='black')
    ax.minorticks_on()
    ax.grid(which='minor', linestyle=':', linewidth='0.5', color='gray')
    ax.set_xlim(Er_list[0],Er_list[-1])
    ax.set_ylim(-1e-2,1e-2)
    # ax.set_yscale('symlog', linthresh=1e-5,linscale=0.1)
    
    ax.set_xlabel(rf'$\omega_r$')
    ax.set_ylabel(r'$\chi_{j}$')
    ax.legend(loc = 'upper left')


In [ ]:

import scqubits

n_evals = 20


# EJ = 3
# EC = 0.6
# EL = 0.13
qbt = scqubits.Fluxonium(EJ = 2.65,EC = 0.6,EL = 0.13, cutoff = 110,flux = 0,truncated_dim=20)
eigenvals = qbt.eigenvals(n_evals)
elements =  qbt.matrixelement_table(operator = "n_operator",evals_count=n_evals)
Er_list = np.linspace(1,8,800)

fig = plt.figure(figsize=(10, 5)) 
gs = gridspec.GridSpec(2, 1, height_ratios=[0.7, 1], hspace=0.15)

plot_fluxonium_transitions(plt.subplot(gs[0]), elements, eigenvals,xlim = (Er_list[0],Er_list[-1]))


# chi0_list,chi1_list,chi2_list = get_chi_lists(qbt,Er_list)
# with open(f'pickles/chi_lists.pkl', 'wb') as file:
#     pickle.dump((chi0_list,chi1_list,chi2_list), file)
with open(f'pickles/chi_lists.pkl', 'rb') as file:
    (chi0_list,chi1_list,chi2_list) = pickle.load(file)

plot_sweep_Er_diagonalization(plt.subplot(gs[1]), chi0_list,chi1_list,chi2_list, Er_list =Er_list)

plt.tight_layout()
plt.savefig('fig02_matrix_elements_and_shift_2.65_0.6_0.13.pdf', format='pdf', bbox_inches='tight')

plt.show()